# Preparação dos Dados — Concatenação Dev + Staging

Esta notebook junta os dois exports sintéticos disponíveis (`employment-with-offer-dev-synthetic.csv`
e `employment-with-offer-staging-synthetic.csv`) em um único arquivo,
`employment-with-offer-synthetic.csv`, usado pelas notebooks de análise
(`nb_employment_offer_analysis.ipynb` e `nb_analise_descritiva.ipynb`).

As duas fontes têm exatamente as mesmas colunas, mas foram geradas de forma independente: os
valores de `id` colidem entre os dois arquivos (mesmo `id`, registros completamente diferentes)
porque são apenas identificadores sintéticos gerados separadamente em cada ambiente — não
representam o mesmo evento de contratação duplicado. Por isso a concatenação é uma simples
união de linhas, sem deduplicação por `id`.

## 1. Configuração & Carregamento dos Dados

In [ ]:
pip install pandas

In [ ]:
import pandas as pd

print('Libraries ready.')

In [ ]:
# ── Google Colab: montar o Drive se estiver rodando lá ─────────────────────
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    # ↓ ajuste para o local onde você colocou os arquivos no Drive
    DATA_DIR = '/content/drive/MyDrive/datascience/files'
else:
    # local / qualquer outro kernel
    DATA_DIR = './data'

DEV_PATH = f'{DATA_DIR}/employment-with-offer-dev-synthetic.csv'
STAGING_PATH = f'{DATA_DIR}/employment-with-offer-staging-synthetic.csv'
OUTPUT_PATH = f'{DATA_DIR}/employment-with-offer-synthetic.csv'

print(f'Dev path      : {DEV_PATH}')
print(f'Staging path  : {STAGING_PATH}')
print(f'Output path   : {OUTPUT_PATH}')

In [ ]:
df_dev = pd.read_csv(DEV_PATH, low_memory=False, on_bad_lines='warn')
df_staging = pd.read_csv(STAGING_PATH, low_memory=False, on_bad_lines='warn')

print(f'Dev       →  {df_dev.shape[0]:,} rows  ×  {df_dev.shape[1]} columns')
print(f'Staging   →  {df_staging.shape[0]:,} rows  ×  {df_staging.shape[1]} columns')

## 2. Verificação de Compatibilidade das Colunas

As duas fontes precisam ter exatamente as mesmas colunas para a concatenação ser direta — sem reindexação implícita ou colunas faltantes viradas em `NaN` por engano.

In [ ]:
assert list(df_dev.columns) == list(df_staging.columns), (
    'As colunas de dev e staging não são idênticas — revisar antes de concatenar.'
)
print(f'{len(df_dev.columns)} colunas idênticas em ambos os arquivos.')

## 3. Concatenação

União simples das linhas (`axis=0`), preservando todas as colunas e todos os registros de
ambas as fontes. O `id` não é usado como chave de deduplicação — ver nota da §título.

In [ ]:
df = pd.concat([df_dev, df_staging], axis=0, ignore_index=True)

print(f'Concatenado  →  {df.shape[0]:,} rows  ×  {df.shape[1]} columns')
assert df.shape[0] == df_dev.shape[0] + df_staging.shape[0]
assert df.shape[1] == df_dev.shape[1] == df_staging.shape[1]

## 4. Exportação

In [ ]:
df.to_csv(OUTPUT_PATH, index=False)
print(f'Salvo em {OUTPUT_PATH}  →  {df.shape[0]:,} rows  ×  {df.shape[1]} columns')